<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c05-header.png" alt="Nextia Learning · Machine Learning: From Problem to Reliable Model" width="100%">

# Solution: Rank a call list

**[Rank a call list](https://learning.nextia-ai.com/courses/ml/m09/rank-a-call-list/)** · Machine Learning: From Problem to Reliable Model · Module 9, case study 4 of 7 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will** rank a marketing call list with a decision tree end to end: find a leak, choose the depth for the team's capacity, read the tree as rules, measure lift, test once, and see what happens when the economy changes.

| | |
|---|---|
| **Time** | About 2 hours with the lesson page. The notebook runs in about 1 minute. |
| **Needs** | An internet connection for the setup cells. No account. The first cell installs pandas 3.0.6 and scikit-learn 1.9.1. |
| **You should know** | Modules 1 to 8 of this course, most of all [Decision trees](https://learning.nextia-ai.com/courses/ml/m05/decision-trees/), [Control complexity](https://learning.nextia-ai.com/courses/ml/m05/control-complexity/), [The precision–recall trade-off](https://learning.nextia-ai.com/courses/ml/m03/precision-recall-trade-off/) and [Plan production checks](https://learning.nextia-ai.com/courses/ml/m08/plan-production-checks/). |
| **Data** | UCI *Bank Marketing* (Moro, Rita and Cortez, 2014), file bank-additional-full, CC BY 4.0, https://doi.org/10.24432/C5K306. Real, anonymous phone calls of a Portuguese bank, 2008–2010. |
| **Tested** | Python 3.14.6 with pandas 3.0.6, scikit-learn 1.9.1 and matplotlib 3.11.2, in Jupyter on macOS, from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/ml/m09/rank-a-call-list/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C05/M09-L04-rank-a-call-list/rank-a-call-list-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs pandas 3.0.6 and scikit-learn 1.9.1, the versions that the course uses, and matplotlib for the charts. It needs an internet connection and takes up to a minute the first time.

You should see `pandas 3.0.6` and `scikit-learn 1.9.1`. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.0.6 scikit-learn==1.9.1 matplotlib
import pandas; print("pandas", pandas.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)

## Setup: get the data

Run the next cell. It downloads `bank_additional_full.csv.gz` (about 0.3 MB) from the Nextia labs repository into a folder `call-list`, and checks its checksum. It needs an internet connection and no account. If a copy of the file is next to this notebook, the cell uses that copy.

You should see `Ready: bank_additional_full.csv.gz, checksum OK.`

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `call-list` (in Colab: the **Files** panel on the left), then run the cell again.

The data is real and anonymous: 41,188 marketing calls of a Portuguese bank, May 2008 to November 2010. Data: "Bank Marketing" by S. Moro, P. Rita and P. Cortez, UCI Machine Learning Repository, https://doi.org/10.24432/C5K306, CC BY 4.0. Nextia Learning saved bank-additional-full.csv with commas and gzip-compressed it.

In [ ]:
# Setup: download the data file into call-list/ and check it. Standard library only.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

URL = "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C05/case-studies/data/bank_additional_full.csv.gz"
FILE, SHA256 = "bank_additional_full.csv.gz", "baad13c38bee0bff5279e68e1f32a8dc60084c2fb05600cd40a5a2ea4d2ce850"
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
HERE, WORK = Path.cwd(), Path("call-list").resolve()
WORK.mkdir(exist_ok=True)
(WORK / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

target = WORK / FILE
if not (target.exists() and sha256(target) == SHA256):
    if (HERE / FILE).exists():
        shutil.copy(HERE / FILE, target)
    else:
        try:
            urllib.request.urlretrieve(URL, target)
        except OSError:  # Python without certificates (python.org on macOS): try curl
            if subprocess.run(["curl", "-fsSL", "-o", str(target), URL]).returncode:
                raise SystemExit(f"Could not download {URL}. Check your internet connection.")
if sha256(target) != SHA256:
    raise SystemExit(f"{FILE} has the wrong checksum. Delete the folder call-list and run again.")
os.chdir(WORK)  # work in this folder: the experiment log and the model are saved here
print(f"Ready: {FILE}, checksum OK.")

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

It also loads pandas as `pd`, numpy as `np` and matplotlib as `plt`, and sets pandas to show 4 decimals. Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it.
import hashlib, json, warnings
import numpy as np
import pandas as pd
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 30)
warnings.filterwarnings("ignore", category=FutureWarning)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")
import matplotlib.pyplot as plt
pd.set_option("display.width", 160)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The situation

Inês Duarte runs the call centre of a bank. Her agents sell a term deposit by phone. This campaign's list is long, and her team can make calls to only **10% of it**. A call that ends with "no" costs about ten minutes of an agent's time; a client who would have said yes but is never called is a lost deposit. So she needs the list **in order**: the most likely clients first. Her analyst, Tiago, builds the model. The [lesson page](https://learning.nextia-ai.com/courses/ml/m09/rank-a-call-list/) tells the full story; this notebook runs every step.

## 2. The data

Load the file and look at it.

In [ ]:
raw = pd.read_csv("bank_additional_full.csv.gz")
print(raw.shape)
print(raw.columns.tolist())

> **Check.** You should see `(41188, 21)`: facts about the client, about the last call, about earlier campaigns, five economic indicators, and the target `y`.

**What this tells us.** One row is the last call to one client in one campaign. The target `y` says whether the client subscribed.

The first checks: missing values, repeated rows, the yes rate, the "unknown" answers and the code 999.

In [ ]:
print("missing values:", int(raw.isna().sum().sum()), "| exact repeats:", int(raw.duplicated().sum()))
print("yes:", int((raw["y"] == "yes").sum()), f"({(raw['y'] == 'yes').mean():.1%})")
text = raw.select_dtypes(include=["object", "string"]).columns
print("unknown:", {c: int((raw[c] == "unknown").sum()) for c in text if (raw[c] == "unknown").any()})
print("pdays = 999:", f"{(raw['pdays'] == 999).mean():.1%}")

> **Check.** You should see `missing values: 0 | exact repeats: 12`, `yes: 4640 (11.3%)`, the "unknown" counts (`default` 8,597, the most), and `pdays = 999: 96.3%`.

**What this tells us.** "No missing values" is not true: the missing answers are written as "unknown". We keep "unknown" as a category, because "the bank does not know" can itself say something. `pdays = 999` means "never contacted before"; a model must not read it as 999 days. And only 1 call in 9 ends with yes.

> **Your turn.** What share of the calls has `default` equal to "unknown"? Put it in `share_unknown_default`, rounded to 3 decimals.

In [ ]:
share_unknown_default = round((raw["default"] == "unknown").mean(), 3)
print(share_unknown_default)

In [ ]:
expect_hash('share_unknown_default', share_unknown_default, '4eb83c23e3c4dcf9')

## 3. Frame the problem

`prepare()` drops the 12 exact repeats, adds a 0/1 target `subscribed`, and replaces the code 999 with a yes/no column `contacted_before`. The rows stay in the file's order, which is time order. Then we name the columns. `duration`, the length of the call, is **not** in `FEATURES`: section 4 shows why.

In [ ]:
def prepare(raw):
    """Drop exact repeats, add the 0/1 target, and replace the code 999 of pdays
    with a yes/no column. The rows stay in the file's order, which is time order."""
    calls = raw.drop_duplicates().reset_index(drop=True)
    calls["subscribed"] = (calls["y"] == "yes").astype(int)
    calls["contacted_before"] = (calls["pdays"] != 999).astype(int)
    return calls


calls = prepare(raw)
CATEGORIES = ["job", "marital", "education", "default", "housing", "loan", "contact", "month", "day_of_week", "poutcome"]
MACRO = ["emp.var.rate", "cons.price.idx", "cons.conf.idx", "euribor3m", "nr.employed"]  # the economy
NUMBERS = ["age", "campaign", "previous", "contacted_before"] + MACRO
FEATURES = CATEGORIES + NUMBERS
TARGET = "subscribed"
print(len(calls), "calls,", len(FEATURES), "features")

> **Check.** You should see `41176 calls, 19 features`.

## 4. Explore what the decision needs: the leak

Tiago's first model used every column and looked excellent. Look at the yes rate by the length of the call.

> **Predict.** Calls under one minute: what share ends with yes?

In [ ]:
duration_band = pd.cut(calls["duration"], [-1, 60, 120, 180, 300, 600, 5000],
                       labels=["0-1 min", "1-2 min", "2-3 min", "3-5 min", "5-10 min", "10+ min"])
print(calls.groupby(duration_band, observed=True)[TARGET].agg(calls="size", yes_rate="mean").round(3))

> **Check.** You should see 0.000 for calls under one minute, rising to 0.486 for calls of 10 minutes or more.

**What this tells us.** A long call goes with a yes: the agent fills in the forms. But the length of a call is known only **after** the call, when the answer is known too. A model that uses it cannot rank a list before anyone calls. It is a **leak**. The source of the data says the same.

In [ ]:
ax = calls.groupby(duration_band, observed=True)[TARGET].mean().plot.bar(color="#3b6fb6", rot=0)
ax.set_xlabel("Length of the call")
ax.set_ylabel("Share of calls that end with yes")
plt.show()

One fact that **is** known before the call: did the client say yes in an earlier campaign? Run the cell.

In [ ]:
print(calls.groupby("poutcome")[TARGET].agg(calls="size", yes_rate="mean").round(3))

> **Check.** `success` (yes in an earlier campaign): 1,373 calls, yes rate 0.651. `nonexistent` (no earlier campaign): 0.088.

**What this tells us.** "Call the earlier yeses first" is an obvious rule. It is the baseline the model must beat. But it covers only 3.3% of the list, less than the team's 10%.

## 5. Split and baselines

The file is in time order, but it has no date column. Inês's question is "rank the rest of **this** campaign's list", so the main split is random and stratified: train 60%, validation 20%, test 20%. Section 10 asks the harder question, "what about the **next** period?", with a time split. Both checks are planned now, before any model.

`hit_rate()` is the measure for the decision: the share of yes among the calls the team can make, the top 10% of the list by score. It is precision at a capacity, as in [The precision–recall trade-off](https://learning.nextia-ai.com/courses/ml/m03/precision-recall-trade-off/). **Lift** is the hit rate divided by the yes rate of the whole list.

In [ ]:
from sklearn.model_selection import train_test_split

rest, test = train_test_split(calls, test_size=0.2, stratify=calls[TARGET], random_state=0)
train, valid = train_test_split(rest, test_size=0.25, stratify=rest[TARGET], random_state=0)
for name, part in [("train", train), ("valid", valid), ("test", test)]:
    print(f"{name:5s} {len(part):6d} calls, {part[TARGET].sum():5d} yes ({part[TARGET].mean():.1%})")

CAPACITY = 0.10


def hit_rate(y, scores, capacity=CAPACITY):
    """Share of yes among the calls with the highest scores, for a capacity share of the list.
    Ties keep the list order (the sort is stable)."""
    calls_made = int(round(capacity * len(y)))
    order = np.argsort(-np.asarray(scores, dtype=float), kind="stable")
    return float(np.asarray(y)[order[:calls_made]].mean())

> **Check.** You should see 24,705, 8,235 and 8,236 calls, each 11.3% yes.

Now the baselines on the validation set: "always no", a random order, and the rule.

> **Predict.** What is the accuracy of a model that always says no?

In [ ]:
y_valid = valid[TARGET].to_numpy()
rule_valid = (valid["poutcome"] == "success").astype(int)
print("calls the team can make:", int(round(CAPACITY * len(valid))))
print("always no: accuracy", round(1 - y_valid.mean(), 3), "| catches", 0, "yes")
print("random order: hit rate", round(y_valid.mean(), 3))
print("rule, earlier success first: hit rate", round(hit_rate(y_valid, rule_valid), 3))

> **Check.** 824 calls; always no: accuracy **0.887**; random order: 0.113; the rule: **0.273**.

**What this tells us.** 88.7% accuracy and zero deposits: accuracy is useless here. The rule more than doubles the random hit rate, but after the 266 earlier yeses it calls the rest in list order. The model must beat 0.273.

## 6. Models and the experiment log

Two pipelines. The tree gets one-hot categories (so its rules read "contact is cellular", not "contact ≤ 0.5") and the numbers as they are: a tree needs no scaling. Logistic regression gets one-hot categories and scaled numbers. The log has the usual columns of this course, plus `cv_top10` and `valid_top10` for the decision's measure. Run the cell. It prints nothing.

In [ ]:
from datetime import date
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, make_scorer, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier


def build_tree(max_depth=None, min_samples_leaf=1, numbers=NUMBERS):
    prepare_step = ColumnTransformer(
        [("categories", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORIES)],
        remainder="passthrough", verbose_feature_names_out=False)
    return Pipeline([("prepare", prepare_step), ("model", DecisionTreeClassifier(
        max_depth=max_depth, min_samples_leaf=min_samples_leaf, random_state=0))])


def build_logistic(numbers=NUMBERS):
    prepare_step = ColumnTransformer([("categories", OneHotEncoder(handle_unknown="ignore"), CATEGORIES),
                                      ("numbers", StandardScaler(), numbers)])
    return Pipeline([("prepare", prepare_step), ("model", LogisticRegression(max_iter=2000))])


cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
top10 = make_scorer(hit_rate, response_method="predict_proba")
LOG = Path("experiment_log.csv")
COLUMNS = ["id", "date", "change", "cv_ap", "cv_std", "cv_top10", "valid_ap", "valid_top10", "decision", "reason"]
pd.DataFrame(columns=COLUMNS).to_csv(LOG, index=False)


def cv_scores(pipeline, columns=FEATURES):
    """AP and the top-10% hit rate on each of the 5 folds of train, and AP on the training folds."""
    return cross_validate(pipeline, train[columns], train[TARGET], cv=cv, return_train_score=True,
                          scoring={"ap": "average_precision", "top10": top10})


def log_experiment(id, change, ap, top, decision, reason):
    row = {"id": id, "date": date.today().isoformat(), "change": change, "cv_ap": round(ap.mean(), 3),
           "cv_std": round(ap.std(), 3), "cv_top10": round(top.mean(), 3), "valid_ap": None, "valid_top10": None,
           "decision": decision, "reason": reason}
    pd.DataFrame([row]).to_csv(LOG, mode="a", header=False, index=False)
    print(f"{id}  CV AP {ap.mean():.3f} ± {ap.std():.3f}  top 10% {top.mean():.3f}  {decision}: {change}")

The first rows: the two baselines, then a tree **with** `duration`, to measure the leak. (A random order has an expected AP and hit rate equal to the yes rate.)

In [ ]:
base = np.array([train[TARGET].iloc[rows].mean() for _, rows in cv.split(train, train[TARGET])])
log_experiment("E01", "Baseline: random order", base, base, "baseline", "The floor: lift 1.")
rule_ap, rule_top = [], []
for _, rows in cv.split(train, train[TARGET]):
    part = train.iloc[rows]
    rule_ap.append(average_precision_score(part[TARGET], part["poutcome"] == "success"))
    rule_top.append(hit_rate(part[TARGET].to_numpy(), (part["poutcome"] == "success").astype(int)))
log_experiment("E02", "Baseline: earlier success first", np.array(rule_ap), np.array(rule_top), "baseline",
               "What the team would do without a model.")
leak = cv_scores(build_tree(None, 100, NUMBERS + ["duration"]), FEATURES + ["duration"])
log_experiment("E03", "Tree, min leaf 100, WITH duration", leak["test_ap"], leak["test_top10"], "reject",
               "Leak: the length of a call is known only after the call.")

> **Check.** E01: 0.113 and 0.113. E02: AP 0.215, top 10% 0.284. E03: AP **0.614**, top 10% 0.616.

**What this tells us.** With the leak, the tree looks twice as good as the rule. That number can never happen in real use, so the row stays in the log as a warning, with `reject`.

Next, a tree with no limits, and the depth sweep of [Control complexity](https://learning.nextia-ai.com/courses/ml/m05/control-complexity/).

> **Predict.** The tree with no limits: will its train AP be high or low? Its CV AP?

In [ ]:
full = cv_scores(build_tree())
print("no limit: train AP", round(full["train_ap"].mean(), 3), "| CV AP", round(full["test_ap"].mean(), 3))
log_experiment("E04", "Tree, no limits", full["test_ap"], full["test_top10"], "reject", "It memorises train.")
sweep = []
for depth in [1, 2, 3, 4, 5, 6, 8, 10, 12, 15]:
    r = cv_scores(build_tree(depth))
    sweep.append({"depth": depth, "train_ap": r["train_ap"].mean(), "cv_ap": r["test_ap"].mean(),
                  "cv_top10": r["test_top10"].mean()})
sweep = pd.DataFrame(sweep).set_index("depth")
print(sweep.round(3))

> **Check.** No limits: train AP 0.999, CV AP 0.178. In the sweep, CV AP is highest at depth 6 (0.405) and falls after it, while train AP keeps rising to 0.784 at depth 15. The top-10% hit rate is close to 0.48 from depth 3 to 6.

**What this tells us.** The full tree learns every training call by heart and fails on new ones. A limit helps. But the two measures disagree on how much: AP, which looks at the whole list, likes depth 5 or 6; the hit rate in the top 10%, which is Inês's decision, is flat from depth 3. The chart shows the gap between train and CV opening.

In [ ]:
ax = sweep[["train_ap", "cv_ap"]].plot(marker="o")
ax.set_xlabel("max_depth")
ax.set_ylabel("Average precision")
plt.show()

Now a grid search over depth and minimum leaf size, scored with both measures. It takes about 20 seconds.

In [ ]:
from sklearn.model_selection import GridSearchCV

search = GridSearchCV(build_tree(), {"model__max_depth": [2, 3, 4, 5, 6, 8, None],
                                     "model__min_samples_leaf": [1, 20, 50, 100, 200]},
                      scoring={"ap": "average_precision", "top10": top10}, refit="top10", cv=cv)
search.fit(train[FEATURES], train[TARGET])
grid = pd.DataFrame(search.cv_results_)[["param_model__max_depth", "param_model__min_samples_leaf", "mean_test_ap",
                                         "mean_test_top10", "std_test_top10"]]
grid.columns = ["max_depth", "min_leaf", "cv_ap", "cv_top10", "cv_top10_std"]
print("best by AP:\n", grid.sort_values("cv_ap", ascending=False).head(3).round(3).to_string(index=False))
print("best by top 10%:\n", grid.sort_values("cv_top10", ascending=False).head(3).round(3).to_string(index=False))

> **Check.** Best by AP: no depth limit with a minimum leaf of 100 (CV AP 0.433, top 10% 0.483). Best by the top 10%: depth 8 with a minimum leaf of 50 (0.496, standard deviation 0.023).

**What this tells us.** The best setting depends on the measure, and the differences are small. Use the [one-standard-error rule](https://learning.nextia-ai.com/courses/ml/m06/tune-hyperparameters/#the-one-standard-error-rule) on the decision's measure: take the simplest tree whose top-10% hit rate is within one standard error of the best.

> **Your turn.** Compute the limit: the best `cv_top10` minus its `cv_top10_std` divided by √5. Then find the smallest `max_depth` in the grid whose `cv_top10` reaches the limit. Put them in `limit` (rounded to 3 decimals) and `chosen_depth`.

In [ ]:
best = grid.sort_values("cv_top10", ascending=False).iloc[0]
limit = round(best["cv_top10"] - best["cv_top10_std"] / np.sqrt(5), 3)
chosen_depth = int(grid[grid["cv_top10"] >= limit]["max_depth"].dropna().min())
print(limit, chosen_depth)

In [ ]:
expect_hash('(limit, chosen_depth)', (limit, chosen_depth), 'a65172a4a1e12b96')

**What this tells us.** A depth-3 tree (top 10%: 0.487) is inside the limit, just. A tree of depth 2 (0.468) is not. Depth 3 means at most 8 leaves: 8 rules that Inês's team can read. Log the candidates, with logistic regression as a check.

In [ ]:
for id, name, pipe, decision, reason in [
        ("E05", "Tree, no depth limit, min leaf 100 (best AP)", build_tree(None, 100), "reject",
         "Best AP, but 183 leaves, and its top 10% (0.483) is below the one-SE limit."),
        ("E06", "Tree, max_depth 8, min leaf 50 (best top 10%)", build_tree(8, 50), "reject",
         "Within the noise of depth 3, with many more leaves."),
        ("E07", "Tree, max_depth 3", build_tree(3), "keep", "Simplest tree within one SE at the capacity: 8 rules."),
        ("E08", "Logistic regression, 19 features", build_logistic(), "reject",
         "Same top 10% (0.489); the team asked for rules it can read.")]:
    r = cv_scores(pipe)
    log_experiment(id, name, r["test_ap"], r["test_top10"], decision, reason)

> **Check.** E05 0.433 | 0.483, E06 0.420 | 0.496, E07 0.371 | **0.487**, E08 0.448 | 0.489.

Confirm the choice on the validation set, once, for the chosen tree and two candidates.

In [ ]:
candidates = {"E05": build_tree(None, 100), "E07": build_tree(3), "E08": build_logistic()}
log = pd.read_csv(LOG)
for id, pipe in candidates.items():
    pipe.fit(train[FEATURES], train[TARGET])
    s = pipe.predict_proba(valid[FEATURES])[:, 1]
    log.loc[log["id"] == id, ["valid_ap", "valid_top10"]] = [round(average_precision_score(y_valid, s), 3),
                                                             round(hit_rate(y_valid, s), 3)]
    print(f"{id}  ROC AUC {roc_auc_score(y_valid, s):.3f}  AP {average_precision_score(y_valid, s):.3f}"
          f"  top 10% {hit_rate(y_valid, s):.3f}  lift {hit_rate(y_valid, s) / y_valid.mean():.2f}")
log.to_csv(LOG, index=False)
tree = candidates["E07"]
scores = tree.predict_proba(valid[FEATURES])[:, 1]
print(log[["id", "change", "cv_ap", "cv_top10", "valid_top10", "decision"]].to_string(index=False))

> **Check.** E05: top 10% 0.513; **E07: ROC AUC 0.757, AP 0.389, top 10% 0.510, lift 4.52**; E08: 0.508. Then the log, E01 to E08.

**What this tells us.** At the team's capacity, the 8-leaf tree finds as many yeses as the 183-leaf tree and logistic regression: about 51 in every 100 calls, 4.5 times the random rate. Its AP is lower because it ranks the **bottom** of the list worse, and nobody calls the bottom of the list.

## 7. Read the tree as rules

`export_text()` prints the tree. `weights` are the training calls in each leaf: [no, yes].

In [ ]:
from sklearn.tree import export_text

names = list(tree.named_steps["prepare"].get_feature_names_out())
print(export_text(tree.named_steps["model"], feature_names=names, show_weights=True, decimals=2))

> **Check.** The first question is `nr.employed <= 5087.65`. The leaf with the most yes is `nr.employed <= 5087.65`, `contacted_before > 0.50`, `day_of_week_mon <= 0.50`: weights [161.00, 450.00], about 74% yes. The largest leaf has [18925.00, 1099.00], about 5% yes.

**What this tells us.** The tree's first question is not about the client: `nr.employed` is the number of people employed in Portugal, a quarterly figure. It splits the calls into **periods** of the economy. Within the low-employment period, clients contacted in an earlier campaign said yes about 3 times in 4. The lesson page writes the 8 rules in plain words. Remember the first question for section 11.

## 8. Lift at the capacity

What if Inês gets more agents? The hit rate and the share of all yeses found, for several capacities:

In [ ]:
for capacity in [0.05, 0.10, 0.15, 0.20, 0.30]:
    rate = hit_rate(y_valid, scores, capacity)
    found = rate * round(capacity * len(y_valid)) / y_valid.sum()
    print(f"call {capacity:.0%}: hit rate {rate:.3f}, lift {rate / y_valid.mean():.2f}, finds {found:.1%} of all yes")

> **Check.** 10%: hit rate 0.510, lift 4.52, finds 45.3% of all yes. 20%: 0.347, lift 3.08, 61.6%.

**What this tells us.** The first calls are the best ones; each extra 5% of the list adds fewer yeses. Doubling the team from 10% to 20% finds 16 percentage points more of the yeses, at a hit rate of 0.347.

> **Your turn: change one thing.** In a busy week the team can call only **5%** of the list. Put the hit rate of the tree on valid at 5% in `rate_5`, rounded to 3 decimals. Predict first: higher or lower than at 10%?

In [ ]:
rate_5 = round(hit_rate(y_valid, scores, 0.05), 3)
print(rate_5)

In [ ]:
expect_hash('rate_5', rate_5, 'a8db99af28345a7d')

## 9. Errors and slices

Which yeses does the team never call? Each valid call falls into one leaf. Count the missed yeses (not in the top 10%) by leaf.

In [ ]:
order = np.argsort(-scores, kind="stable")
called = np.zeros(len(valid), dtype=bool)
called[order[: int(round(CAPACITY * len(valid)))]] = True
leaf = tree.named_steps["model"].apply(tree.named_steps["prepare"].transform(valid[FEATURES]))
errors = pd.DataFrame({"leaf": leaf, "score": scores.round(3), "called": called, "yes": y_valid,
                       "missed_yes": (y_valid == 1) & ~called})
print(errors.groupby(["leaf", "score"]).agg(calls=("yes", "size"), called=("called", "sum"), yes=("yes", "sum"),
                                            missed_yes=("missed_yes", "sum")))

> **Check.** The largest leaf has 6,703 calls with a score of 0.055: none is called, and it holds 360 of the 508 missed yeses. The leaf with score 0.38 is called only in part (542 of its 612 calls).

**What this tells us.** Most missed yeses sit in the big "normal economy" leaf, where about 1 call in 19 succeeds. The tree cannot sort those 6,703 calls: they all get the same score. That is the price of 8 rules. A finer model could rank inside the leaf, but at 10% capacity the team never reaches it.

Now the people on the list. A ranking decides who gets a sales call, so check it by age band.

In [ ]:
age_band = pd.cut(valid["age"], [16, 25, 40, 60, 100], labels=["17-25", "26-40", "41-60", "61+"])
frame = pd.DataFrame({"age band": age_band.to_numpy(), "called": called, "yes": y_valid})
slices = frame.groupby("age band", observed=True).agg(clients=("yes", "size"), yes_rate=("yes", "mean"),
                                                      share_called=("called", "mean"))
slices["hit rate in calls"] = frame[frame["called"]].groupby("age band", observed=True)["yes"].mean()
print(slices.round(3))

> **Check.** 61+: 179 clients, yes rate 0.441, **69.3%** of them on the call list. 26–40: 4,431 clients, 8.8% called.

**What this tells us.** Older clients said yes far more often, so the ranking calls most of them. That is efficient, but a list that targets older clients for a financial product needs care: the product must suit them, and the calls must not pressure them. The lesson page discusses it.

## 10. The final test, once

The tree (E07) and the capacity (10%) are fixed. Open the test set once, and compare with the rule and a random order on the same list.

> **Warning.** Run the next cell once. Do not change the tree after you read it.

In [ ]:
y_test = test[TARGET].to_numpy()
test_scores = tree.predict_proba(test[FEATURES])[:, 1]
calls_made = int(round(CAPACITY * len(test)))
rule_test = (test["poutcome"] == "success").astype(int)
print("calls:", calls_made, "| ROC AUC", round(roc_auc_score(y_test, test_scores), 3),
      "| AP", round(average_precision_score(y_test, test_scores), 3))
for name, rate in [("tree, depth 3", hit_rate(y_test, test_scores)), ("rule", hit_rate(y_test, rule_test)),
                   ("random (expected)", y_test.mean())]:
    print(f"{name:18s} hit rate {rate:.3f}  lift {rate / y_test.mean():.2f}  yes in {calls_made} calls: {rate * calls_made:.0f}")

> **Check.** 824 calls. The tree: hit rate **0.478**, lift **4.24**, 394 yes. The rule: 0.263, 217 yes. Random: about 93 yes.

**What this tells us.** On calls the tree never saw, 824 calls find about 394 deposits, against 217 with the rule and 93 at random. The test is a little lower than validation (0.478 against 0.510): validation helped to choose, so it was a little optimistic.

## 11. Epilogue: the next period

The random split mixed all periods. Now the harder question, planned in section 5: train on the first 80% of the file (earlier calls), and test on the last 20% (later calls). First, how does the file change over time?

In [ ]:
tenth = np.arange(len(calls)) * 10 // len(calls) + 1
by_tenth = calls.groupby(tenth).agg(yes_rate=(TARGET, "mean"), euribor3m=("euribor3m", "mean"),
                                    employed=("nr.employed", "mean"))
print(by_tenth.round(3))

> **Check.** The yes rate rises from 0.028 in the first tenth to 0.460 in the last; the Euribor rate falls from about 4.86 to 0.80.

**What this tells us.** The economy changed, and so did the clients' answers. The economic columns are **clocks**: they tell the model which period a call is from. Now fit on the early calls and score the late ones.

> **Predict.** Will the tree's hit rate in the top 10% of the late calls be higher or lower than 0.478? Logistic regression?

In [ ]:
cut = int(len(calls) * 0.8)
early, late = calls.iloc[:cut], calls.iloc[cut:]
y_late = late[TARGET].to_numpy()
print("yes rate: early", round(early[TARGET].mean(), 3), "| late", round(y_late.mean(), 3))
for name, pipe in [("tree, depth 3", build_tree(3)), ("logistic regression", build_logistic())]:
    s = pipe.fit(early[FEATURES], early[TARGET]).predict_proba(late[FEATURES])[:, 1]
    print(f"{name:20s} ROC AUC {roc_auc_score(y_late, s):.3f}  top 10% {hit_rate(y_late, s):.3f}"
          f"  lift {hit_rate(y_late, s) / y_late.mean():.2f}  mean score {s.mean():.3f}")
print("late calls with a Euribor rate below anything in early:",
      f"{(late['euribor3m'] < early['euribor3m'].min()).mean():.1%}")

> **Check.** Yes rate 0.064 early and 0.308 late. The tree: ROC AUC **0.638**, top 10% 0.416, lift 1.35, mean score 0.125. Logistic regression: ROC AUC **0.748**, top 10% 0.570, mean score 0.256. 98.5% of the late calls have a Euribor rate below anything in the early data.

**What this tells us.** A tree can only answer with the leaves it learned. Almost every late call has economic values the tree never saw, so the calls fall into a few leaves with old rates: a mean score of 0.125 when 30.8% say yes. Logistic regression follows the trend of the numbers and holds up better. But both lose: this is **drift**. The answer is monitoring and retraining, not a perfect model. The lesson page turns this into a plan.

## Next

You have ranked a call list end to end: a leak found and removed, a depth chosen for the capacity, 8 rules the team can read, lift on one honest test, and the limit that drift sets. The [lesson page](https://learning.nextia-ai.com/courses/ml/m09/rank-a-call-list/) has the rules in plain words, the recommendations, the monitoring plan, the portfolio template and the knowledge checks.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Choose a model for hospital readmissions](https://learning.nextia-ai.com/courses/ml/m09/choose-a-model-for-readmissions/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/ml/m09/rank-a-call-list/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The data is real and anonymous, from the UCI Machine Learning Repository (CC BY 4.0).